# 🌿 Mapudungun AI Translator
## Neural Machine Translation: Spanish ↔ Mapudungun

---

### What This Is
A **Neural Machine Translation (NMT)** model - NOT an LLM.  
Think **Google Translate**, not ChatGPT.

### Features
- **Multiple data sources**: AVENUE Corpus, Glosbe, or Combined
- **Continue training**: Load existing model and train further
- **API-friendly**: Clean function calls for all operations

### Before You Start
1. **Runtime → Change runtime type → T4 GPU**
2. Have your HuggingFace token ready (Write access)

---

## ⚙️ CONFIGURATION

**Set your preferences here before running:**

In [ ]:
#@title ⚙️ Configuration { display-mode: "form" }
#@markdown ### Data Source
DATA_SOURCE = "avenue" #@param ["avenue", "glosbe", "combined"]
#@markdown - **avenue**: AVENUE Corpus (~260K pairs, conversations, recommended)
#@markdown - **glosbe**: Glosbe scraping (dictionary style, smaller, slower)
#@markdown - **combined**: Both sources merged (best coverage)

#@markdown ---
#@markdown ### Training Mode
TRAINING_MODE = "continue" #@param ["fresh", "continue"]
#@markdown - **fresh**: Train from base NLLB-200 model
#@markdown - **continue**: Load existing checkpoint and train further

#@markdown ---
#@markdown ### Model Settings
CHECKPOINT_PATH = "" #@param {type:"string"}
#@markdown Leave empty for fresh training. For continue: `username/model-name` or local path

#@markdown ---
#@markdown ### Training Parameters
NUM_EPOCHS = 3 #@param {type:"slider", min:1, max:10, step:1}
BATCH_SIZE = 4 #@param {type:"slider", min:1, max:8, step:1}
LEARNING_RATE = 2e-4 #@param {type:"number"}
MAX_GLOSBE_WORDS = 500 #@param {type:"slider", min:100, max:2000, step:100}
#@markdown (Only used if DATA_SOURCE includes glosbe)

#@markdown ---
#@markdown ### Output Settings
HF_REPO_NAME = "mapudungun-translator" #@param {type:"string"}
PUSH_TO_HUB = True #@param {type:"boolean"}

#@markdown ---
# Print configuration
print("="*50)
print("📋 CONFIGURATION SUMMARY")
print("="*50)
print(f"Data Source:     {DATA_SOURCE}")
print(f"Training Mode:   {TRAINING_MODE}")
print(f"Checkpoint:      {CHECKPOINT_PATH or 'None (fresh)'}")
print(f"Epochs:          {NUM_EPOCHS}")
print(f"Batch Size:      {BATCH_SIZE}")
print(f"Learning Rate:   {LEARNING_RATE}")
print(f"Push to Hub:     {PUSH_TO_HUB}")
print("="*50)

## 📦 Step 1: Setup Environment

In [ ]:
#@title 1.1 Check GPU
import torch

def check_gpu():
    """Verify GPU is available and return info."""
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU Available: {gpu_name}")
        print(f"✅ GPU Memory: {gpu_memory:.1f} GB")
        return True
    else:
        print("❌ No GPU found!")
        print("👉 Go to Runtime → Change runtime type → Select 'T4 GPU'")
        return False

check_gpu()

In [ ]:
#@title 1.2 Install Dependencies
%%capture
!pip install -q transformers>=4.36.0 peft>=0.7.0 datasets>=2.15.0
!pip install -q accelerate>=0.25.0 bitsandbytes>=0.41.0 sentencepiece>=0.1.99
!pip install -q beautifulsoup4>=4.12.0 requests>=2.31.0 tqdm>=4.66.0
!pip install -q sacrebleu>=2.3.0 huggingface_hub

print("✅ All packages installed!")

In [ ]:
#@title 1.3 Login to HuggingFace
from huggingface_hub import login, HfApi

def hf_login():
    """Login to HuggingFace Hub."""
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        print("✅ Found token in Colab Secrets!")
    except:
        print("Enter your HuggingFace token (Write access required):")
        token = input()

    login(token=token)
    api = HfApi()
    username = api.whoami()["name"]
    print(f"✅ Logged in as: {username}")
    return username

HF_USERNAME = hf_login()

## 📚 Step 2: Data Loading API

Functions to load data from different sources.

In [ ]:
#@title 2.1 AVENUE Corpus Loader
import os
import re
import requests
from tqdm import tqdm

def load_avenue_corpus(verbose=True):
    """
    Load the AVENUE Mapudungun-Spanish corpus from GitHub.
    
    The corpus format has M: (Mapudungun) and C: (Castellano/Spanish) lines.

    Returns:
        list: List of {'source': spanish, 'target': mapudungun} dicts
    """
    if verbose:
        print("📥 Downloading AVENUE Corpus from GitHub...")

    # GitHub API to list files in translation-clean directory
    api_url = "https://api.github.com/repos/mingjund/mapudungun-corpus/contents/translation-clean"
    raw_base = "https://raw.githubusercontent.com/mingjund/mapudungun-corpus/master/translation-clean"

    translations = []

    try:
        # Get list of files
        response = requests.get(api_url, timeout=30)
        if response.status_code != 200:
            print(f"❌ Failed to get file list: {response.status_code}")
            return []

        files = response.json()
        txt_files = [f['name'] for f in files if f['name'].endswith('.txt')]

        if verbose:
            print(f"   Found {len(txt_files)} translation files")

        # Download and parse each file
        for filename in tqdm(txt_files, desc="Downloading files", disable=not verbose):
            try:
                file_url = f"{raw_base}/{filename}"
                file_response = requests.get(file_url, timeout=30)

                if file_response.status_code == 200:
                    content = file_response.text
                    
                    # Parse M: and C: pairs
                    # Format: M: mapudungun text\nC: spanish text
                    current_mapudungun = None
                    
                    for line in content.split('\n'):
                        line = line.strip()
                        
                        # Skip comments and empty lines
                        if not line or line.startswith(';'):
                            continue
                        
                        # Check for M: (Mapudungun) line
                        if line.startswith('M:'):
                            current_mapudungun = line[2:].strip()
                            # Clean up annotations like <uh>, <*SPA>, etc.
                            current_mapudungun = re.sub(r'<[^>]+>', '', current_mapudungun).strip()
                        
                        # Check for C: (Castellano/Spanish) line
                        elif line.startswith('C:') and current_mapudungun:
                            spanish = line[2:].strip()
                            # Clean up annotations
                            spanish = re.sub(r'<[^>]+>', '', spanish).strip()
                            
                            # Add pair if both are non-empty
                            if current_mapudungun and spanish and len(current_mapudungun) > 1 and len(spanish) > 1:
                                translations.append({
                                    'source': spanish,
                                    'target': current_mapudungun
                                })
                            
                            current_mapudungun = None

            except Exception as e:
                if verbose:
                    print(f"   ⚠️ Error processing {filename}: {e}")
                continue

        # Deduplicate
        seen = set()
        unique = []
        for t in translations:
            key = (t['source'].lower(), t['target'].lower())
            if key not in seen:
                seen.add(key)
                unique.append(t)

        if verbose:
            print(f"\n✅ AVENUE Corpus loaded: {len(unique)} unique pairs")

        return unique

    except Exception as e:
        print(f"❌ Error loading AVENUE corpus: {e}")
        return []

# Test the function
print("AVENUE Corpus Loader ready.")
print("Usage: data = load_avenue_corpus()")

In [ ]:
#@title 2.2 Glosbe Scraper
import requests
from bs4 import BeautifulSoup
import time
import re
import random
from tqdm import tqdm

class GlosbeScraper:
    """
    Scrapes Spanish-Mapudungun translations from Glosbe.com

    ⚠️ WARNING: Glosbe's API is shut down. Scraping may violate ToS.
    Use responsibly with delays. Consider AVENUE corpus instead.
    """

    def __init__(self):
        self.base_url = "https://glosbe.com"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })
        self.translations = []

    def _get_page(self, url, retries=3):
        """Fetch page with retry logic and rate limiting."""
        for attempt in range(retries):
            try:
                time.sleep(random.uniform(2.0, 4.0))  # Be respectful
                response = self.session.get(url, timeout=15)
                if response.status_code == 200:
                    return BeautifulSoup(response.text, 'html.parser')
                elif response.status_code == 429:
                    print(f"⚠️ Rate limited, waiting 60 seconds...")
                    time.sleep(60)
            except Exception as e:
                if attempt == retries - 1:
                    print(f"⚠️ Failed after {retries} attempts: {e}")
                time.sleep(5)
        return None

    def _get_words_for_letter(self, letter, lang_from='es', lang_to='arn'):
        """Get words starting with a letter."""
        url = f"{self.base_url}/{lang_from}/{lang_to}/similar/{letter}"
        soup = self._get_page(url)
        words = []
        if soup:
            links = soup.find_all('a', href=re.compile(f'/{lang_from}/{lang_to}/'))
            for link in links:
                word = link.get_text(strip=True)
                if word and len(word) > 1 and not word.startswith('http'):
                    words.append(word)
        return list(set(words))

    def _get_translations_for_word(self, word, lang_from='es', lang_to='arn'):
        """Get translations for a word."""
        url = f"{self.base_url}/{lang_from}/{lang_to}/{requests.utils.quote(word)}"
        soup = self._get_page(url)
        results = []

        if not soup:
            return results

        # Find translations
        for div in soup.find_all('div', class_=re.compile('translation|example|tmem')):
            text = div.get_text(strip=True)
            if text and text != word and len(text) < 500:
                results.append({'source': word, 'target': text})

        return results

    def scrape(self, max_words=500, verbose=True):
        """
        Scrape translations from Glosbe.

        Args:
            max_words: Maximum words to process
            verbose: Print progress

        Returns:
            list: List of {'source': spanish, 'target': mapudungun} dicts
        """
        if verbose:
            print("🌐 Starting Glosbe scraper...")
            print("⚠️ This may take 15-30+ minutes. Be patient!")
            print("⚠️ Consider using AVENUE corpus instead (faster, legal).")

        # Get words from alphabet
        all_words = []
        alphabet = 'abcdefghijklmnopqrstuvwxyz'

        for letter in tqdm(alphabet, desc="Scanning alphabet", disable=not verbose):
            words = self._get_words_for_letter(letter)
            all_words.extend(words)

        all_words = list(set(all_words))
        if verbose:
            print(f"📝 Found {len(all_words)} unique words")

        # Limit words
        if len(all_words) > max_words:
            all_words = random.sample(all_words, max_words)

        # Scrape translations
        for word in tqdm(all_words, desc="Scraping translations", disable=not verbose):
            translations = self._get_translations_for_word(word)
            self.translations.extend(translations)

        # Deduplicate
        seen = set()
        unique = []
        for t in self.translations:
            key = (t['source'].lower(), t['target'].lower())
            if key not in seen:
                seen.add(key)
                unique.append(t)

        if verbose:
            print(f"✅ Scraped {len(unique)} unique pairs")

        return unique


def load_glosbe_data(max_words=500, verbose=True):
    """
    Load data from Glosbe via scraping.

    Args:
        max_words: Maximum words to scrape
        verbose: Print progress

    Returns:
        list: List of {'source': spanish, 'target': mapudungun} dicts
    """
    scraper = GlosbeScraper()
    return scraper.scrape(max_words=max_words, verbose=verbose)


print("Glosbe Scraper ready.")
print("Usage: data = load_glosbe_data(max_words=500)")

In [ ]:
#@title 2.3 Combined Data Loader

def load_training_data(source='avenue', max_glosbe_words=500, verbose=True):
    """
    Load training data from specified source(s).

    Args:
        source: 'avenue', 'glosbe', or 'combined'
        max_glosbe_words: Max words for Glosbe scraping
        verbose: Print progress

    Returns:
        list: List of {'source': spanish, 'target': mapudungun} dicts
    """
    data = []

    if source in ['avenue', 'combined']:
        if verbose:
            print("\n" + "="*50)
            print("📚 Loading AVENUE Corpus...")
            print("="*50)
        avenue_data = load_avenue_corpus(verbose=verbose)
        data.extend(avenue_data)
        if verbose:
            print(f"   AVENUE: {len(avenue_data)} pairs")

    if source in ['glosbe', 'combined']:
        if verbose:
            print("\n" + "="*50)
            print("🌐 Loading Glosbe Data...")
            print("="*50)
        glosbe_data = load_glosbe_data(max_words=max_glosbe_words, verbose=verbose)
        data.extend(glosbe_data)
        if verbose:
            print(f"   Glosbe: {len(glosbe_data)} pairs")

    # Deduplicate combined data
    if source == 'combined':
        seen = set()
        unique = []
        for item in data:
            key = (item['source'].lower(), item['target'].lower())
            if key not in seen:
                seen.add(key)
                unique.append(item)
        data = unique

    if verbose:
        print("\n" + "="*50)
        print(f"✅ Total data loaded: {len(data)} pairs")
        print("="*50)

    return data


print("Combined Data Loader ready.")
print("Usage: data = load_training_data(source='avenue')")
print("       data = load_training_data(source='glosbe', max_glosbe_words=500)")
print("       data = load_training_data(source='combined')")

In [ ]:
#@title 2.4 Data Cleaning & Preparation
import re
import random

def clean_text(text):
    """Clean and normalize text."""
    if not text or not isinstance(text, str):
        return None
    text = ' '.join(text.split())
    if len(text) > 500 or len(text) < 2:
        return None
    if re.match(r'^[\d\s\.,;:!?]+$', text):
        return None
    return text


def clean_data(data, verbose=True):
    """
    Clean and filter translation pairs.

    Args:
        data: List of {'source': str, 'target': str} dicts
        verbose: Print progress

    Returns:
        list: Cleaned data
    """
    if verbose:
        print("🧹 Cleaning data...")

    cleaned = []
    for pair in data:
        source = clean_text(pair.get('source', ''))
        target = clean_text(pair.get('target', ''))

        if not source or not target:
            continue
        if source.lower() == target.lower():
            continue

        len_ratio = len(source) / len(target) if len(target) > 0 else 0
        if len_ratio > 5 or len_ratio < 0.2:
            continue

        cleaned.append({'source': source, 'target': target})

    # Deduplicate
    seen = set()
    unique = []
    for item in cleaned:
        key = (item['source'].lower(), item['target'].lower())
        if key not in seen:
            seen.add(key)
            unique.append(item)

    if verbose:
        print(f"   Before: {len(data)} | After: {len(unique)} | Removed: {len(data) - len(unique)}")

    return unique


def split_data(data, train_ratio=0.8, val_ratio=0.1, seed=42):
    """
    Split data into train/validation/test sets.

    Args:
        data: List of translation pairs
        train_ratio: Training set ratio
        val_ratio: Validation set ratio
        seed: Random seed

    Returns:
        tuple: (train_data, val_data, test_data)
    """
    random.seed(seed)
    shuffled = data.copy()
    random.shuffle(shuffled)

    total = len(shuffled)
    train_size = int(total * train_ratio)
    val_size = int(total * val_ratio)

    train = shuffled[:train_size]
    val = shuffled[train_size:train_size + val_size]
    test = shuffled[train_size + val_size:]

    print(f"📊 Data split: Train={len(train)} | Val={len(val)} | Test={len(test)}")

    return train, val, test


print("Data cleaning functions ready.")

## 🤖 Step 3: Model API

In [ ]:
#@title 3.1 Model Loading Functions
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel, TaskType
import torch

# Language codes
SPANISH_CODE = "spa_Latn"
MAPUDUNGUN_CODE = "quy_Latn"  # Using Quechua as proxy (agglutinative, closest available)
BASE_MODEL = "facebook/nllb-200-distilled-600M"


def load_base_model(verbose=True):
    """
    Load the base NLLB-200 model with 8-bit quantization.

    Returns:
        tuple: (model, tokenizer)
    """
    if verbose:
        print(f"📥 Loading base model: {BASE_MODEL}")

    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_threshold=6.0
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

    model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL,
        quantization_config=quantization_config,
        device_map="auto",
        dtype=torch.float16
    )

    if verbose:
        print(f"✅ Model loaded!")
        print(f"   GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    return model, tokenizer


def load_checkpoint(checkpoint_path, verbose=True):
    """
    Load a previously trained model checkpoint.

    Args:
        checkpoint_path: HuggingFace repo (user/model) or local path
        verbose: Print progress

    Returns:
        tuple: (model, tokenizer)
    """
    if verbose:
        print(f"📥 Loading checkpoint: {checkpoint_path}")

    # Load base model first
    base_model, tokenizer = load_base_model(verbose=False)

    # Load LoRA adapters
    model = PeftModel.from_pretrained(base_model, checkpoint_path)

    if verbose:
        print(f"✅ Checkpoint loaded!")

    return model, tokenizer


def setup_lora(model, r=16, lora_alpha=32, lora_dropout=0.1, verbose=True):
    """
    Configure LoRA adapters for efficient training.

    Args:
        model: Base model
        r: LoRA rank
        lora_alpha: LoRA alpha
        lora_dropout: Dropout rate
        verbose: Print progress

    Returns:
        model: Model with LoRA adapters
    """
    if verbose:
        print("⚙️ Configuring LoRA...")

    # Prepare model for k-bit training (disable gradient checkpointing to avoid conflicts)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
    
    # Disable cache for training
    model.config.use_cache = False

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type=TaskType.SEQ_2_SEQ_LM,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "fc1", "fc2"]
    )

    model = get_peft_model(model, lora_config)
    
    # Ensure LoRA parameters require gradients
    for name, param in model.named_parameters():
        if 'lora' in name.lower():
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    if verbose:
        print(f"✅ LoRA configured!")
        print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

    return model


print("Model API ready.")
print("Usage: model, tokenizer = load_base_model()")
print("       model, tokenizer = load_checkpoint('username/model')")
print("       model = setup_lora(model)")

In [ ]:
#@title 3.2 Training API
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset
import os


def prepare_dataset(data, tokenizer, max_length=128):
    """
    Prepare data for training (bidirectional).

    Args:
        data: List of {'source': spanish, 'target': mapudungun} dicts
        tokenizer: Model tokenizer
        max_length: Maximum sequence length

    Returns:
        Dataset: Tokenized HuggingFace dataset
    """
    # Create bidirectional data
    bidirectional = []
    for pair in data:
        # Spanish -> Mapudungun
        bidirectional.append({
            'source_text': pair['source'],
            'target_text': pair['target'],
            'source_lang': SPANISH_CODE,
            'target_lang': MAPUDUNGUN_CODE
        })
        # Mapudungun -> Spanish
        bidirectional.append({
            'source_text': pair['target'],
            'target_text': pair['source'],
            'source_lang': MAPUDUNGUN_CODE,
            'target_lang': SPANISH_CODE
        })

    def tokenize(example):
        tokenizer.src_lang = example['source_lang']
        inputs = tokenizer(
            example['source_text'],
            max_length=max_length,
            truncation=True,
            padding='max_length'
        )

        tokenizer.src_lang = example['target_lang']
        labels = tokenizer(
            example['target_text'],
            max_length=max_length,
            truncation=True,
            padding='max_length'
        )

        inputs['labels'] = [
            -100 if t == tokenizer.pad_token_id else t
            for t in labels['input_ids']
        ]
        return inputs

    dataset = Dataset.from_list(bidirectional)
    tokenized = dataset.map(tokenize, remove_columns=dataset.column_names)

    print(f"📊 Dataset prepared: {len(tokenized)} samples (bidirectional)")
    return tokenized


def train_model(
    model,
    tokenizer,
    train_dataset,
    val_dataset=None,
    output_dir="./mapudungun-translator",
    num_epochs=3,
    batch_size=4,
    learning_rate=2e-4,
    push_to_hub=False,
    hub_model_id=None,
    backup_branch="backup",
    verbose=True
):
    """
    Train the model.

    Args:
        model: Model to train
        tokenizer: Tokenizer
        train_dataset: Training dataset
        val_dataset: Validation dataset (optional)
        output_dir: Output directory
        num_epochs: Number of training epochs
        batch_size: Batch size
        learning_rate: Learning rate
        push_to_hub: Auto-backup to HuggingFace during training
        hub_model_id: HuggingFace repo (e.g., 'username/model-name')
        backup_branch: Branch name for backups (e.g., 'backup-avenue')
        verbose: Print progress

    Returns:
        trainer: Trained Trainer object
    """
    os.makedirs(output_dir, exist_ok=True)

    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=4,
        learning_rate=learning_rate,
        warmup_steps=100,
        weight_decay=0.01,
        eval_strategy="steps" if val_dataset else "no",
        eval_steps=200 if val_dataset else None,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        logging_steps=50,
        fp16=True,
        optim="adamw_torch",
        predict_with_generate=True,
        generation_max_length=128,
        load_best_model_at_end=True if val_dataset else False,
        report_to="tensorboard",
        # Auto-backup to HuggingFace (backup branch)
        push_to_hub=push_to_hub,
        hub_model_id=hub_model_id if push_to_hub else None,
        hub_strategy="checkpoint" if push_to_hub else None,
        hub_revision=backup_branch if push_to_hub else None,
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    if verbose:
        print("🚀 Starting training...")
        print(f"   Epochs: {num_epochs} | Batch: {batch_size} | LR: {learning_rate}")
        if push_to_hub:
            print(f"   Auto-backup: ON → {hub_model_id} (branch: {backup_branch})")

    result = trainer.train()

    if verbose:
        print(f"\n✅ Training complete!")
        print(f"   Loss: {result.training_loss:.4f}")
        print(f"   Time: {result.metrics['train_runtime']/60:.1f} min")

    return trainer


print("Training API ready.")
print("Usage: dataset = prepare_dataset(data, tokenizer)")
print("       trainer = train_model(model, tokenizer, train_ds, val_ds)")

In [ ]:
#@title 3.3 Save & Push API
from huggingface_hub import HfApi, create_repo


def save_model(trainer, tokenizer, output_dir, train_info=None):
    """
    Save model locally with README.

    Args:
        trainer: Trained Trainer object
        tokenizer: Tokenizer
        output_dir: Output directory
        train_info: Dict with training info for README
    """
    print("💾 Saving model locally...")

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Create README
    info = train_info or {}
    readme = f'''---
language: [es, arn]
license: apache-2.0
tags: [translation, mapudungun, spanish, nmt, nllb, lora]
pipeline_tag: translation
---

# Mapudungun-Spanish Translator (NMT)

**Neural Machine Translation model** - NOT an LLM.
Think Google Translate, not ChatGPT.

## Model Info
- **Type**: Seq2Seq Translation (NMT)
- **Base**: facebook/nllb-200-distilled-600M
- **Method**: LoRA fine-tuning
- **Data source**: {info.get('data_source', 'Unknown')}
- **Training pairs**: {info.get('train_pairs', 'Unknown')}
- **Language proxy**: Quechua (quy_Latn) for Mapudungun (isolate language)

## About Mapudungun
Language of the Mapuche people (Chile/Argentina). A **language isolate** with ~260,000 speakers.

*Pewkayal!* 🌿
'''

    with open(f"{output_dir}/README.md", "w") as f:
        f.write(readme)

    print(f"✅ Saved to {output_dir}")


def push_to_hub(output_dir, repo_name, username):
    """
    Push model to HuggingFace Hub.

    Args:
        output_dir: Local model directory
        repo_name: Repository name
        username: HuggingFace username

    Returns:
        str: Repository URL
    """
    repo_id = f"{username}/{repo_name}"
    print(f"📤 Uploading to: {repo_id}")

    try:
        create_repo(repo_id, exist_ok=True)

        api = HfApi()
        api.upload_folder(
            folder_path=output_dir,
            repo_id=repo_id,
            commit_message="Upload Mapudungun translator"
        )

        url = f"https://huggingface.co/{repo_id}"
        print(f"✅ Uploaded! {url}")
        return url

    except Exception as e:
        print(f"❌ Upload failed: {e}")
        return None


print("Save & Push API ready.")

In [ ]:
#@title 3.4 Translation API
import torch


def translate(text, direction, model, tokenizer, max_length=128):
    """
    Translate text.

    Args:
        text: Text to translate
        direction: 'es_to_arn' or 'arn_to_es'
        model: Translation model
        tokenizer: Tokenizer
        max_length: Max output length

    Returns:
        str: Translated text
    """
    if direction == 'es_to_arn':
        src_lang, tgt_lang = SPANISH_CODE, MAPUDUNGUN_CODE
    else:
        src_lang, tgt_lang = MAPUDUNGUN_CODE, SPANISH_CODE

    tokenizer.src_lang = src_lang

    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=max_length,
        truncation=True,
        padding=True
    ).to(model.device)

    forced_bos = tokenizer.lang_code_to_id.get(tgt_lang, tokenizer.bos_token_id)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos,
            max_length=max_length,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def translate_es_to_arn(text, model, tokenizer):
    """Translate Spanish to Mapudungun."""
    return translate(text, 'es_to_arn', model, tokenizer)


def translate_arn_to_es(text, model, tokenizer):
    """Translate Mapudungun to Spanish."""
    return translate(text, 'arn_to_es', model, tokenizer)


def batch_translate(texts, direction, model, tokenizer):
    """
    Translate multiple texts.

    Args:
        texts: List of texts
        direction: 'es_to_arn' or 'arn_to_es'
        model: Model
        tokenizer: Tokenizer

    Returns:
        list: List of translations
    """
    return [translate(t, direction, model, tokenizer) for t in texts]


print("Translation API ready.")
print("Usage: result = translate_es_to_arn('Hola', model, tokenizer)")
print("       result = translate_arn_to_es('Mari mari', model, tokenizer)")

## 🚀 Step 4: Run Training Pipeline

This uses your configuration from the top.

In [ ]:
#@title 4.1 Load Data (based on your config)
import os
import json
from huggingface_hub import HfApi, hf_hub_download, upload_file, list_repo_refs

#@markdown ⚠️ Force re-download even if cached:
FORCE_RELOAD = False  #@param {type:"boolean"}

DATA_BRANCH = f"data-{DATA_SOURCE}"  # e.g., data-avenue, data-glosbe
CACHE_FILENAME = "data_cache.json"
LOCAL_CACHE = f"./data_cache_{DATA_SOURCE}.json"
REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

api = HfApi()

def data_branch_exists():
    """Check if data branch exists on HuggingFace."""
    try:
        refs = list_repo_refs(REPO_ID)
        return DATA_BRANCH in [b.name for b in refs.branches]
    except:
        return False

def load_from_hub():
    """Load cached data from HuggingFace."""
    path = hf_hub_download(
        repo_id=REPO_ID,
        filename=CACHE_FILENAME,
        revision=DATA_BRANCH,
        local_dir="./"
    )
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_to_hub(data):
    """Save data cache to HuggingFace."""
    # Save locally first
    with open(LOCAL_CACHE, 'w', encoding='utf-8') as f:
        json.dump(data, f)
    
    # Upload to HuggingFace
    api.upload_file(
        path_or_fileobj=LOCAL_CACHE,
        path_in_repo=CACHE_FILENAME,
        repo_id=REPO_ID,
        revision=DATA_BRANCH,
        create_pr=False,
        commit_message=f"Cache {DATA_SOURCE} data ({len(data['train'])} train pairs)"
    )

# Check if data already in memory
if 'train_data' in dir() and len(train_data) > 0 and not FORCE_RELOAD:
    print(f"✅ Data already in memory: {len(train_data)} train pairs")
    print("   (Set FORCE_RELOAD = True to re-download)")

# Check if cached on HuggingFace
elif data_branch_exists() and not FORCE_RELOAD:
    print(f"📂 Loading from HuggingFace: {REPO_ID} (branch: {DATA_BRANCH})")
    try:
        cache = load_from_hub()
        train_data = cache['train']
        val_data = cache['val']
        test_data = cache['test']
        print(f"✅ Loaded: {len(train_data)} train, {len(val_data)} val, {len(test_data)} test")
        print("   (Set FORCE_RELOAD = True to re-download)")
    except Exception as e:
        print(f"⚠️ Cache load failed: {e}")
        print("   Will download fresh...")
        FORCE_RELOAD = True

# Download fresh
if FORCE_RELOAD or 'train_data' not in dir():
    print(f"📚 Downloading data from: {DATA_SOURCE}")
    print("="*50)

    # Load raw data
    raw_data = load_training_data(
        source=DATA_SOURCE,
        max_glosbe_words=MAX_GLOSBE_WORDS,
        verbose=True
    )

    # Clean data
    clean = clean_data(raw_data, verbose=True)

    # Split data
    train_data, val_data, test_data = split_data(clean)

    # Cache to HuggingFace
    print(f"\n💾 Caching to HuggingFace (branch: {DATA_BRANCH})...")
    try:
        save_to_hub({'train': train_data, 'val': val_data, 'test': test_data})
        print("   ✅ Cached! Next time will load in seconds.")
    except Exception as e:
        print(f"   ⚠️ Could not cache: {e}")

print(f"\n✅ Data ready! Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

In [ ]:
#@title 4.2 Load/Setup Model (based on your config)

print(f"🤖 Training mode: {TRAINING_MODE}")
print("="*50)

if TRAINING_MODE == "continue":
    checkpoint = CHECKPOINT_PATH if CHECKPOINT_PATH else f"{HF_USERNAME}/{HF_REPO_NAME}"
    print(f"📥 Loading checkpoint: {checkpoint}")
    try:
        model, tokenizer = load_checkpoint(checkpoint)
    except Exception as e:
        error_msg = str(e)
        if "Repository Not Found" in error_msg or "Can't find" in error_msg or "404" in error_msg:
            print("\n" + "="*60)
            print("\n⚠️  NO CHECKPOINT FOUND!\n")
            print(f'You have TRAINING_MODE = "continue", which tells the notebook')
            print(f"to download your previous training checkpoint from HuggingFace.")
            print(f'\nBut the repository "{checkpoint}" either doesn\' + "'" + 't exist or is')
            print(f"empty — no previous training has been saved there yet.")
            print("\n" + "="*60)
            print(f"\n👉 Would you like to start fresh training instead?")
            user_choice = input("   Type 'yes' to start fresh, or anything else to stop: ").strip().lower()
            if user_choice in ['yes', 'y', 'si', 'sí']:
                print("\n📥 Loading base model (fresh training)...")
                model, tokenizer = load_base_model()
                model = setup_lora(model)
            else:
                raise RuntimeError("Training stopped. Set TRAINING_MODE to \"fresh\" in the config cell if needed.") from None
        else:
            raise
else:
    print("📥 Loading base model...")
    model, tokenizer = load_base_model()
    model = setup_lora(model)

print("\n✅ Model ready!")


In [ ]:
#@title 4.3 Prepare Datasets

print("📊 Preparing datasets...")
print("="*50)

train_dataset = prepare_dataset(train_data, tokenizer)
val_dataset = prepare_dataset(val_data, tokenizer) if val_data else None

print("\n✅ Datasets ready!")

In [ ]:
#@title 4.4 Train! 🚀

# Memory cleanup (safe to re-run this cell)
import gc
import torch
if 'trainer' in dir():
    del trainer
gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = "./mapudungun-translator"
HUB_MODEL_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
BACKUP_BRANCH = f"backup-{DATA_SOURCE}"  # e.g., backup-avenue, backup-glosbe

print("🚀 STARTING TRAINING")
print("="*50)
print(f"Data source: {DATA_SOURCE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Auto-backup: {'ON → ' + HUB_MODEL_ID + ' (branch: ' + BACKUP_BRANCH + ')' if PUSH_TO_HUB else 'OFF'}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("="*50)
print("☕ This will take 30-60+ minutes...")
print()

trainer = train_model(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID,
    backup_branch=BACKUP_BRANCH
)

In [ ]:
#@title 4.5 Save & Upload (Official Release to main branch)

# Save locally
save_model(
    trainer=trainer,
    tokenizer=tokenizer,
    output_dir=OUTPUT_DIR,
    train_info={
        'data_source': DATA_SOURCE,
        'train_pairs': len(train_data)
    }
)

# Push to Hub (main branch - official release)
if PUSH_TO_HUB:
    print("📤 Pushing OFFICIAL release to main branch...")
    push_to_hub(OUTPUT_DIR, HF_REPO_NAME, HF_USERNAME)

## 🧪 Step 5: Test Translations

In [ ]:
#@title 5.1 Quick Tests

print("🧪 Testing Spanish → Mapudungun")
print("="*50)

test_es = ["Hola", "Buenos días", "¿Cómo estás?", "Gracias", "Te quiero"]

for text in test_es:
    result = translate_es_to_arn(text, model, tokenizer)
    print(f"  {text} → {result}")

print("\n🧪 Testing Mapudungun → Spanish")
print("="*50)

# Use samples from test data
test_arn = [pair['target'] for pair in test_data[:5]] if test_data else ["Mari mari"]

for text in test_arn:
    result = translate_arn_to_es(text, model, tokenizer)
    print(f"  {text} → {result}")

In [ ]:
#@title 5.2 Interactive Translator

print("🎮 Interactive Translator")
print("="*50)
print("Format: 'es: texto' or 'arn: texto'")
print("Type 'quit' to exit\n")

while True:
    user_input = input(">>> ").strip()

    if user_input.lower() == 'quit':
        print("👋 Pewkayal!")
        break

    if user_input.startswith('es:'):
        text = user_input[3:].strip()
        result = translate_es_to_arn(text, model, tokenizer)
        print(f"    → {result}")
    elif user_input.startswith('arn:'):
        text = user_input[4:].strip()
        result = translate_arn_to_es(text, model, tokenizer)
        print(f"    → {result}")
    else:
        print("    ⚠️ Use 'es:' or 'arn:' prefix")

## 📖 API Reference

### Data Loading
```python
# Load from specific source
data = load_training_data(source='avenue')      # AVENUE corpus
data = load_training_data(source='glosbe')      # Glosbe scraping
data = load_training_data(source='combined')    # Both

# Clean and split
clean = clean_data(data)
train, val, test = split_data(clean)
```

### Model Loading
```python
# Fresh model
model, tokenizer = load_base_model()
model = setup_lora(model)

# Continue from checkpoint
model, tokenizer = load_checkpoint('username/model-name')
```

### Training
```python
# Prepare data
train_ds = prepare_dataset(train_data, tokenizer)
val_ds = prepare_dataset(val_data, tokenizer)

# Train (without auto-backup)
trainer = train_model(
    model, tokenizer, train_ds, val_ds,
    num_epochs=3, batch_size=4, learning_rate=2e-4
)

# Train (with auto-backup to HuggingFace)
trainer = train_model(
    model, tokenizer, train_ds, val_ds,
    num_epochs=3, batch_size=4, learning_rate=2e-4,
    push_to_hub=True,
    hub_model_id='username/model-name'
)

# Save
save_model(trainer, tokenizer, './output')
push_to_hub('./output', 'model-name', 'username')
```

### Translation
```python
# Single
result = translate_es_to_arn('Hola', model, tokenizer)
result = translate_arn_to_es('Mari mari', model, tokenizer)

# Batch
results = batch_translate(['Hola', 'Gracias'], 'es_to_arn', model, tokenizer)
```

In [ ]:
#@title 🗑️ Delete Backup & Data Branches (Optional Cleanup)

#@markdown ⚠️ Enable deletion:
DELETE_BACKUPS = False  #@param {type:"boolean"}
DELETE_DATA_CACHE = False  #@param {type:"boolean"}

if not DELETE_BACKUPS and not DELETE_DATA_CACHE:
    print("🔒 Deletion disabled.")
    print("   Check DELETE_BACKUPS to remove training backups (backup-*)")
    print("   Check DELETE_DATA_CACHE to remove cached data (data-*)")
else:
    from huggingface_hub import HfApi, list_repo_refs

    api = HfApi()
    REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

    # List all branches
    print(f"📋 Branches in {REPO_ID}:")
    refs = list_repo_refs(REPO_ID)
    
    backup_branches = [b.name for b in refs.branches if b.name.startswith('backup')]
    data_branches = [b.name for b in refs.branches if b.name.startswith('data')]
    
    to_delete = []
    
    if DELETE_BACKUPS and backup_branches:
        print("\n🔄 Backup branches (training checkpoints):")
        for branch in backup_branches:
            print(f"   - {branch}")
        to_delete.extend(backup_branches)
    elif DELETE_BACKUPS:
        print("\n🔄 No backup branches found.")
    
    if DELETE_DATA_CACHE and data_branches:
        print("\n📦 Data cache branches:")
        for branch in data_branches:
            print(f"   - {branch}")
        to_delete.extend(data_branches)
    elif DELETE_DATA_CACHE:
        print("\n📦 No data cache branches found.")
    
    if to_delete:
        # Safety confirmation
        print(f"\n⚠️ This will DELETE {len(to_delete)} branch(es) above!")
        confirm = input("Type 'DELETE' to confirm: ")
        
        if confirm == 'DELETE':
            print("\n🗑️ Deleting branches...")
            for branch in to_delete:
                try:
                    api.delete_branch(repo_id=REPO_ID, branch=branch)
                    print(f"   ✅ Deleted '{branch}'")
                except Exception as e:
                    print(f"   ⚠️ Could not delete '{branch}': {e}")
            print("\n✅ Cleanup complete! Main branch untouched.")
        else:
            print("\n❌ Cancelled. No branches deleted.")
    else:
        print("\n✅ Nothing to delete.")

In [ ]:
#@title 📥 Download Model (Optional)

from google.colab import files
import shutil

print("📦 Creating zip file...")
shutil.make_archive('mapudungun-translator', 'zip', OUTPUT_DIR)

print("📥 Starting download...")
files.download('mapudungun-translator.zip')